# SpatialEval - VLM Analysis

Comprehensive analysis of 11 VLMs on the SpatialEval benchmark (4635 samples/model, VQA mode).

In [ ]:
import json, os, re, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)

sys.path.insert(0, '.')
from evals.evaluation import (
    extract_answer_from_text_spatialmap,
    extract_answer_from_text_mazenav,
    extract_answer_from_text_spatialgrid,
    extract_answer_from_text_spatialreal,
)
from evals.eval_robust import (
    extract_option_letter, match_substring, match_exact,
    match_word_boundary, match_option_letter, MODEL_SHORT,
    TASK_QID_LABELS, DATA_DIR, EXTRACTORS,
)

# Short names and ordered model list
MODEL_ORDER = [
    'Molmo2-8B', 'Qwen3-4B', 'MiniCPM-V-4.5', 'Molmo2-4B',
    'SAIL-VL2-8B', 'Qwen3-8B', 'LLaVA-OV-4B', 'InternVL3-4B',
    'InternVL3-8B', 'LLaVA-OV-8B', 'Gemma3-4B'
]

FAMILY_COLORS = {
    'Molmo2': '#2ecc71', 'Qwen3': '#3498db', 'MiniCPM': '#9b59b6',
    'SAIL': '#1abc9c', 'LLaVA': '#e74c3c', 'InternVL': '#e67e22',
    'Gemma': '#e91e63',
}

def family_of(name):
    for k in FAMILY_COLORS:
        if k in name:
            return k
    return 'Other'

MODEL_PALETTE = {m: FAMILY_COLORS[family_of(m)] for m in MODEL_ORDER}

In [ ]:
# Load all data
DATA_DIR_PATH = DATA_DIR
records = []

for filename in sorted(os.listdir(DATA_DIR_PATH)):
    if not filename.endswith('.jsonl'):
        continue
    model_name = filename.replace('m-', '').replace('_w_reason.jsonl', '')
    short = MODEL_SHORT.get(model_name, model_name)
    with open(os.path.join(DATA_DIR_PATH, filename)) as fh:
        for line in fh:
            item = json.loads(line)
            sid = item['id']
            parts = sid.split('.')
            task, qid = parts[0], int(parts[-1])
            raw = item.get('answer', '')
            oracle = item.get('oracle_answer', '')
            opt = item.get('oracle_option', '')

            ext_fn = EXTRACTORS.get(task)
            try:
                extracted = ext_fn(raw, qid, model_name) if ext_fn else None
            except Exception:
                extracted = None

            ext_str = str(extracted).lower().strip() if extracted is not None else ''
            oracle_str = str(oracle).lower().strip() if oracle else ''

            records.append({
                'model': short, 'sample_id': sid, 'task': task, 'qid': qid,
                'task_qid': TASK_QID_LABELS.get((task, qid), f'{task}:q{qid}'),
                'oracle': oracle_str, 'oracle_option': (opt or '').upper(),
                'extracted': ext_str,
                'raw': raw[:300] if raw else '',
                'substring': match_substring(oracle_str, ext_str),
                'exact': match_exact(oracle_str, ext_str),
                'word_boundary': match_word_boundary(oracle or '', ext_str),
                'option_letter': match_option_letter(raw, opt or ''),
            })

df = pd.DataFrame(records)
print(f'Loaded {len(df)} samples, {df["model"].nunique()} models')
df.head()

## 1. Results Overview

In [ ]:
# 1a: Overall accuracy bar chart
overall = df.groupby('model')['exact'].mean().reindex(MODEL_ORDER)

fig, ax = plt.subplots(figsize=(10, 5))
colors = [MODEL_PALETTE.get(m, '#999') for m in overall.index]
bars = ax.barh(range(len(overall)), overall.values * 100, color=colors)
ax.set_yticks(range(len(overall)))
ax.set_yticklabels(overall.index)
ax.invert_yaxis()
ax.set_xlabel('Accuracy (%)')
ax.set_title('Overall Accuracy (Exact Match)')
for i, v in enumerate(overall.values):
    ax.text(v * 100 + 0.5, i, f'{v*100:.1f}%', va='center', fontsize=9)
ax.set_xlim(0, 72)
plt.tight_layout()
plt.savefig('analysis_figures/01_overall_accuracy.png', dpi=150)
plt.show()

In [ ]:
# 1b: Heatmap - accuracy per model x task
pivot = df.groupby(['model', 'task'])['exact'].mean().unstack()
pivot = pivot.reindex(MODEL_ORDER)[['spatialmap', 'mazenav', 'spatialgrid', 'spatialreal']]

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(pivot * 100, annot=True, fmt='.1f', cmap='YlOrRd',
            ax=ax, vmin=0, vmax=90, linewidths=0.5)
ax.set_ylabel('')
ax.set_title('Accuracy (%) per Model x Task (Exact Match)')
plt.tight_layout()
plt.savefig('analysis_figures/02_heatmap_task.png', dpi=150)
plt.show()

## 2. Per-Question-Type Breakdown

In [ ]:
# 2a: Grouped bar charts per task
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
tasks = ['spatialmap', 'mazenav', 'spatialgrid', 'spatialreal']

for ax, task in zip(axes.flat, tasks):
    tdf = df[df['task'] == task]
    acc = tdf.groupby(['model', 'qid'])['exact'].mean().reset_index()
    acc['qid_label'] = acc['qid'].astype(str)
    # Sort models by overall accuracy
    model_order_t = acc.groupby('model')['exact'].mean().sort_values(ascending=False).index.tolist()
    acc['model'] = pd.Categorical(acc['model'], categories=model_order_t, ordered=True)

    sns.barplot(data=acc, x='qid_label', y='exact', hue='model',
                palette=MODEL_PALETTE, ax=ax, hue_order=model_order_t)
    ax.set_title(f'{task}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Question ID')
    ax.set_ylabel('Accuracy')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
    ax.legend(fontsize=6, ncol=3, loc='upper right')

plt.suptitle('Accuracy by Question Type within Each Task', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('analysis_figures/03_per_question_type.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 2b: Detailed accuracy table
acc_detail = df.groupby(['model', 'task_qid'])['exact'].mean().unstack()
acc_detail = acc_detail.reindex(MODEL_ORDER)
cols_ordered = [TASK_QID_LABELS.get((t, q), '') for t in ['spatialmap','mazenav','spatialgrid','spatialreal'] for q in range(3)]
cols_present = [c for c in cols_ordered if c in acc_detail.columns]
acc_detail = acc_detail[cols_present]
acc_detail.style.format('{:.1%}').highlight_max(axis=0, color='lightgreen')

## 3. Answer Extraction Comparison

In [ ]:
# 3a: Side-by-side strategy comparison
strategies = ['substring', 'exact', 'word_boundary', 'option_letter']
strat_acc = df.groupby('model')[strategies].mean().reindex(MODEL_ORDER) * 100

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(MODEL_ORDER))
width = 0.18
for i, s in enumerate(strategies):
    ax.bar(x + i * width, strat_acc[s], width, label=s.replace('_', ' ').title())
ax.set_xticks(x + 1.5 * width)
ax.set_xticklabels(MODEL_ORDER, rotation=35, ha='right')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Evaluation Strategy Comparison')
ax.legend()
ax.set_ylim(0, 75)
plt.tight_layout()
plt.savefig('analysis_figures/04_strategy_comparison.png', dpi=150)
plt.show()

In [ ]:
# 3b: Delta heatmap (substring - exact)
delta = df.groupby(['model', 'task']).agg(
    sub=('substring', 'mean'), exc=('exact', 'mean')
).reset_index()
delta['diff'] = delta['sub'] - delta['exc']
delta_pivot = delta.pivot(index='model', columns='task', values='diff')
delta_pivot = delta_pivot.reindex(MODEL_ORDER)[['spatialmap','mazenav','spatialgrid','spatialreal']]

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(delta_pivot * 100, annot=True, fmt='.1f', cmap='Reds',
            ax=ax, linewidths=0.5, vmin=0, vmax=6)
ax.set_ylabel('')
ax.set_title('Accuracy Inflation: Substring - Exact (percentage points)')
plt.tight_layout()
plt.savefig('analysis_figures/05_delta_heatmap.png', dpi=150)
plt.show()

In [ ]:
# 3c: Disagreement summary
fp = df[df['substring'] & ~df['exact']].copy()
fn = df[~df['substring'] & df['exact']].copy()

print(f'False Positives (substring correct, exact wrong): {len(fp)}')
print(f'False Negatives (substring wrong, exact correct): {len(fn)}')
print()
print('FP breakdown by (model, task):')
if len(fp) > 0:
    for (m, t), grp in fp.groupby(['model', 'task']):
        print(f'  {m:<18} {t:<15} {len(grp)} FP')
        # Show a sample
        sample = grp.iloc[0]
        print(f'    oracle: "{sample["oracle"]}" | extracted: "{sample["extracted"][:80]}"')
        print()

## 4. Error Analysis

In [ ]:
# 4a: Sample errors per task/qid from top-3 models
top_models = ['Molmo2-8B', 'Qwen3-4B', 'MiniCPM-V-4.5']
errors = df[(~df['exact']) & (df['model'].isin(top_models))]

for task in ['spatialmap', 'mazenav', 'spatialgrid', 'spatialreal']:
    tdf = errors[errors['task'] == task]
    if len(tdf) == 0:
        continue
    print(f'\n=== {task.upper()} ERROR SAMPLES (top models) ===')
    for qid in sorted(tdf['qid'].unique()):
        qdf = tdf[tdf['qid'] == qid].head(3)
        print(f'  qid={qid}:')
        for _, row in qdf.iterrows():
            print(f'    [{row["model"]}] oracle="{row["oracle"]}" | got="{row["extracted"][:60]}"')
            print(f'      raw: {row["raw"][:120]}...')
        print()

In [ ]:
# 4b: False positive showcase
print('=== FALSE POSITIVE SHOWCASE ===')
print('Samples where substring matched but exact did not:\n')
for _, row in fp.head(10).iterrows():
    print(f'[{row["model"]}] task={row["task"]} qid={row["qid"]}')
    print(f'  Oracle:    "{row["oracle"]}"')
    print(f'  Extracted: "{row["extracted"][:100]}"')
    print(f'  Match reason: oracle is substring of extracted')
    print()

## 5. Per-Instance Difficulty

In [ ]:
# 5a: Difficulty histogram
# difficulty = fraction of models that got it WRONG
difficulty = df.groupby('sample_id').agg(
    wrong=('exact', lambda x: 1 - x.mean()),
    task=('task', 'first'),
    qid=('qid', 'first'),
    oracle=('oracle', 'first'),
    task_qid=('task_qid', 'first'),
).reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(difficulty['wrong'], bins=np.arange(0, 1.05, 0.1), edgecolor='black', alpha=0.8)
ax.set_xlabel('Difficulty (fraction of models wrong)')
ax.set_ylabel('Number of instances')
ax.set_title('Distribution of Instance Difficulty')
n_easy = (difficulty['wrong'] == 0).sum()
n_hard = (difficulty['wrong'] == 1).sum()
ax.axvline(x=0, color='green', linestyle='--', alpha=0.5, label=f'Easy (all correct): {n_easy}')
ax.axvline(x=1, color='red', linestyle='--', alpha=0.5, label=f'Hard (all wrong): {n_hard}')
ax.legend()
plt.tight_layout()
plt.savefig('analysis_figures/06_difficulty_hist.png', dpi=150)
plt.show()
print(f'Total unique instances: {len(difficulty)}')
print(f'Easy (all 11 models correct): {n_easy} ({n_easy/len(difficulty)*100:.1f}%)')
print(f'Hard (all 11 models wrong): {n_hard} ({n_hard/len(difficulty)*100:.1f}%)')

In [ ]:
# 5b: Top hardest and easiest instances
print('=== TOP 10 HARDEST INSTANCES (all models wrong) ===')
hardest = difficulty[difficulty['wrong'] == 1].head(10)
for _, row in hardest.iterrows():
    print(f'  {row["sample_id"]} | {row["task"]} qid={row["qid"]} | oracle="{row["oracle"]}"')

print(f'\n=== TOP 10 EASIEST INSTANCES (all models correct) ===')
easiest = difficulty[difficulty['wrong'] == 0].head(10)
for _, row in easiest.iterrows():
    print(f'  {row["sample_id"]} | {row["task"]} qid={row["qid"]} | oracle="{row["oracle"]}"')

In [ ]:
# 5c: Difficulty by task/qid box plot
fig, ax = plt.subplots(figsize=(12, 5))
order = [TASK_QID_LABELS.get((t, q), '') for t in ['spatialmap','mazenav','spatialgrid','spatialreal'] for q in range(3)]
order = [o for o in order if o in difficulty['task_qid'].values]
sns.boxplot(data=difficulty, x='task_qid', y='wrong', order=order, ax=ax)
ax.set_xlabel('')
ax.set_ylabel('Difficulty (fraction wrong)')
ax.set_title('Instance Difficulty by Question Type')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('analysis_figures/07_difficulty_boxplot.png', dpi=150)
plt.show()

In [ ]:
# 5d: Difficulty vs oracle answer for numeric tasks
numeric_tasks = [('mazenav', 0), ('mazenav', 1), ('spatialmap', 2), ('spatialgrid', 0)]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, (task, qid) in zip(axes, numeric_tasks):
    tdf = difficulty[(difficulty['task'] == task) & (difficulty['qid'] == qid)].copy()
    tdf['oracle_num'] = pd.to_numeric(tdf['oracle'], errors='coerce')
    tdf = tdf.dropna(subset=['oracle_num'])
    if len(tdf) == 0:
        continue
    # Jittered scatter
    jitter = np.random.uniform(-0.05, 0.05, len(tdf))
    ax.scatter(tdf['oracle_num'] + jitter, tdf['wrong'] + jitter, alpha=0.3, s=15)
    ax.set_xlabel('Oracle Answer (numeric)')
    ax.set_ylabel('Difficulty')
    label = TASK_QID_LABELS.get((task, qid), f'{task}:q{qid}')
    ax.set_title(label, fontsize=10)

plt.suptitle('Difficulty vs Oracle Answer Value', fontsize=13)
plt.tight_layout()
plt.savefig('analysis_figures/08_difficulty_vs_oracle.png', dpi=150)
plt.show()

## 6. Model Correlation Heatmap

In [ ]:
# 6a: Pairwise correctness correlation
correct_matrix = df.pivot_table(index='sample_id', columns='model', values='exact')
correct_matrix = correct_matrix.reindex(columns=MODEL_ORDER)
corr = correct_matrix.corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=0, vmax=1,
            ax=ax, linewidths=0.5)
ax.set_title('Pairwise Correctness Correlation Between Models')
plt.tight_layout()
plt.savefig('analysis_figures/09_model_correlation.png', dpi=150)
plt.show()

In [ ]:
# 6b: Agreement rate heatmap
models = MODEL_ORDER
agree = pd.DataFrame(index=models, columns=models, dtype=float)
for m1 in models:
    for m2 in models:
        a = correct_matrix[m1]
        b = correct_matrix[m2]
        agree.loc[m1, m2] = (a == b).mean()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(agree.astype(float), annot=True, fmt='.2f', cmap='YlGn',
            vmin=0.5, vmax=1, ax=ax, linewidths=0.5)
ax.set_title('Pairwise Agreement Rate (% same answer)')
plt.tight_layout()
plt.savefig('analysis_figures/10_agreement_heatmap.png', dpi=150)
plt.show()

## 7. Confusion Analysis

In [ ]:
# 7a: Direction confusion matrices (spatialmap qid0)
dir_df = df[(df['task'] == 'spatialmap') & (df['qid'] == 0)].copy()
directions = ['northeast', 'northwest', 'southeast', 'southwest']

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for idx, model in enumerate(MODEL_ORDER):
    ax = axes.flat[idx]
    mdf = dir_df[dir_df['model'] == model]
    # Build confusion matrix
    cm = pd.DataFrame(0, index=directions, columns=directions)
    for _, row in mdf.iterrows():
        oracle_d = row['oracle']
        pred_d = row['extracted']
        if oracle_d in directions and pred_d in directions:
            cm.loc[oracle_d, pred_d] += 1
    # Normalize by row
    cm_norm = cm.div(cm.sum(axis=1), axis=0).fillna(0)
    sns.heatmap(cm_norm, annot=True, fmt='.1%', cmap='Blues', ax=ax,
                vmin=0, vmax=1, cbar=False)
    ax.set_title(model, fontsize=10)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Oracle')

# Hide empty subplot
if len(MODEL_ORDER) < 12:
    axes.flat[-1].set_visible(False)

plt.suptitle('Direction Confusion Matrices (spatialmap qid0)', fontsize=14)
plt.tight_layout()
plt.savefig('analysis_figures/11_direction_confusion.png', dpi=150)
plt.show()

In [ ]:
# 7b: Animal confusion matrices (spatialgrid qid1+2)
animal_df = df[(df['task'] == 'spatialgrid') & (df['qid'].isin([1, 2]))].copy()
animals = ['giraffe', 'cat', 'dog', 'elephant', 'rabbit']

fig, axes = plt.subplots(3, 4, figsize=(16, 14))
for idx, model in enumerate(MODEL_ORDER):
    ax = axes.flat[idx]
    mdf = animal_df[animal_df['model'] == model]
    cm = pd.DataFrame(0, index=animals, columns=animals)
    for _, row in mdf.iterrows():
        oracle_a = row['oracle']
        pred_a = row['extracted']
        if oracle_a in animals and pred_a in animals:
            cm.loc[oracle_a, pred_a] += 1
    cm_norm = cm.div(cm.sum(axis=1), axis=0).fillna(0)
    sns.heatmap(cm_norm, annot=True, fmt='.1%', cmap='Greens', ax=ax,
                vmin=0, vmax=1, cbar=False)
    ax.set_title(model, fontsize=10)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Oracle')

if len(MODEL_ORDER) < 12:
    axes.flat[-1].set_visible(False)

plt.suptitle('Animal Confusion Matrices (spatialgrid qid1+2)', fontsize=14)
plt.tight_layout()
plt.savefig('analysis_figures/12_animal_confusion.png', dpi=150)
plt.show()

In [ ]:
# 7c: Aggregate confusion patterns
# Direction
cm_all_dir = pd.DataFrame(0, index=directions, columns=directions)
for _, row in dir_df.iterrows():
    oracle_d, pred_d = row['oracle'], row['extracted']
    if oracle_d in directions and pred_d in directions:
        cm_all_dir.loc[oracle_d, pred_d] += 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm_all_dir, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Aggregate Direction Confusion (all models)')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Oracle')

# Animal
cm_all_animal = pd.DataFrame(0, index=animals, columns=animals)
for _, row in animal_df.iterrows():
    oracle_a, pred_a = row['oracle'], row['extracted']
    if oracle_a in animals and pred_a in animals:
        cm_all_animal.loc[oracle_a, pred_a] += 1

sns.heatmap(cm_all_animal, annot=True, fmt='d', cmap='Greens', ax=axes[1])
axes[1].set_title('Aggregate Animal Confusion (all models)')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Oracle')

plt.tight_layout()
plt.savefig('analysis_figures/13_aggregate_confusion.png', dpi=150)
plt.show()

## 8. Answer Format Analysis

In [ ]:
# Format classification
def classify_format(raw):
    if not raw or not raw.strip():
        return 'empty'
    raw = raw.strip()
    if re.match(r'^[A-D][\.\:]\s', raw):
        return 'option_dot'
    if re.match(r'^[A-D]$', raw):
        return 'bare_letter'
    if re.search(r'(?i)the answer is [A-D]', raw):
        return 'answer_is_X'
    if re.search(r'\*\*Concise Answer:\*\*', raw):
        return 'markdown_concise'
    if re.search(r'\*\*Answer:\*\*', raw):
        return 'markdown_answer'
    return 'free_text'

df['format'] = df['raw'].apply(classify_format)
format_counts = df.groupby(['model', 'format']).size().unstack(fill_value=0)
format_pct = format_counts.div(format_counts.sum(axis=1), axis=0) * 100
format_pct = format_pct.reindex(MODEL_ORDER)

In [ ]:
# 8a: Stacked bar chart of format distribution
fig, ax = plt.subplots(figsize=(12, 5))
format_cols = ['option_dot', 'bare_letter', 'answer_is_X', 'markdown_concise', 'markdown_answer', 'free_text', 'empty']
cols_present = [c for c in format_cols if c in format_pct.columns]
format_pct[cols_present].plot(kind='barh', stacked=True, ax=ax,
    colormap='Set2', edgecolor='white', linewidth=0.5)
ax.set_xlabel('Percentage (%)')
ax.set_ylabel('')
ax.set_title('Answer Format Distribution per Model')
ax.legend(title='Format', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('analysis_figures/14_format_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 8b: Accuracy by format type
format_acc = df.groupby(['model', 'format'])['exact'].mean().unstack()
format_acc = format_acc.reindex(MODEL_ORDER) * 100
cols_present = [c for c in format_cols if c in format_acc.columns]
format_acc[cols_present].style.format('{:.1f}').highlight_max(axis=1, color='lightgreen')